# 🚀 DeepSeek-R1 Model with Azure AI Inference 🧠

**DeepSeek-R1** is a state-of-the-art reasoning model combining reinforcement learning and supervised fine-tuning, excelling at complex reasoning tasks with 37B active parameters and 128K context window.

In this notebook, you'll learn to:
1. **Initialize** the ChatCompletionsClient for Azure serverless endpoints
2. **Chat** with DeepSeek-R1 using reasoning extraction
3. **Implement** a travel planning example with step-by-step reasoning
4. **Leverage** the 128K context window for complex scenarios

## Why DeepSeek-R1?
- **Advanced Reasoning**: Specializes in chain-of-thought problem solving
- **Massive Context**: 128K token window for detailed analysis
- **Efficient Architecture**: 37B active parameters from 671B total
- **Safety Integrated**: Built-in content filtering capabilities


## 1. Setup & Authentication

cred.json file requirements:
```bash
MODEL_NAME=DeepSeek-R1
```

In [12]:
import os
import json
from pathlib import Path

def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

start_dir = os.getcwd()  # Use current directory
file_path = find_cred_json(start_dir)

print(f"Found cred.json at: {file_path}")

try:
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)
    
    print("Project Connection String:", loaded_config['PROJECT_CONNECTION_STRING'])
    print("Tenant ID:", loaded_config['TENANT_ID'])
    print("Model Deployment ID:", loaded_config['MODEL_DEPLOYMENT_NAME'])

except FileNotFoundError:
    print(f"Could not find file at: {file_path}")
except json.JSONDecodeError:
    print(f"File exists but contains invalid JSON")
except TypeError:
    print("File path was None — cred.json not found in any parent directories.")


Found cred.json at: /workspaces/Azure-AI-Foundry-steup/text-gen-ai-foundry-/cred.json
Project Connection String: eastus2.api.azureml.ms;1c2fd79b-ad21-4ad0-8d53-12de16650452;rg-sarath-8734;sarath-8734
Tenant ID: 02e58275-def8-41c4-82c4-f7864c28f7c9
Model Deployment ID: DeepSeek-R1


In [13]:
import os
import re
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Initialize the AI Project client
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=loaded_config["PROJECT_ENDPOINT"],
)

# Get Azure OpenAI client
openai_client = project_client.get_openai_client(
    api_version="2024-10-21"  # Adjust if your Azure environment uses a different API version
)



## 2. Intelligent Travel Planning ✈️

Demonstrate DeepSeek-R1's reasoning capabilities for trip planning:

In [17]:
def plan_trip_with_reasoning(query, show_thinking=False):
    """Get travel recommendations with reasoning extraction."""
    messages = [
        {"role": "system", "content": "You are a travel expert. Provide detailed plans with rationale."},
        {"role": "user", "content": f"{query} Include hidden gems and safety considerations."}
    ]
    
    # Call DeepSeek-R1 reasoning model
    response = openai_client.chat.completions.create(
        model=loaded_config.get("MODEL_NAME", "DeepSeek-R1"),  # Replace with your deployment name
        messages=messages,
        temperature=0.7,
        max_tokens=1024
    )
    
    content = response.choices[0].message.content
    
    # Extract reasoning if present
    if show_thinking:
        match = re.search(r"<think>(.*?)</think>(.*)", content, re.DOTALL)
        if match:
            return {
                "thinking": match.group(1).strip(),
                "answer": match.group(2).strip()
            }
    return content

# Example usage
query = "Plan a 5-day cultural trip to Kyoto in April"
result = plan_trip_with_reasoning(query, show_thinking=True)

print("🗺️ Query:", query)
if isinstance(result, dict):
    print("\n🧠 Thinking Process:", result["thinking"])
    print("\n📝 Final Answer:", result["answer"])
else:
    print("\n📝 Response:", result)


🗺️ Query: Plan a 5-day cultural trip to Kyoto in April

📝 Response: **5-Day Cultural Trip to Kyoto in April: Hidden Gems & Safety Tips**  
*Rationale: April offers mild weather and cherry blossoms (sakura), but crowds are intense. This itinerary balances iconic sites with lesser-known spots, prioritizes efficient transit, and includes safety tips for a seamless experience.*

---

### **Day 1: Arrival & Historic Gion**  
**Morning:**  
- **Arrival:** Land at Kansai International Airport (KIX). Take the *Haruka Express* train to Kyoto Station (75 mins).  
- **Check-in:** Stay in a machiya (traditional townhouse) in **Higashiyama** for cultural immersion.  

**Afternoon:**  
- **Gion District:** Stroll Hanami-koji Street. Skip crowded tea houses; visit **Gion Tatsumi Bridge** for quiet sakura views.  
- **Hidden Gem:** **Kennin-ji Temple** (Kyoto’s oldest Zen temple) with a stunning dragon ceiling and tranquil bamboo garden.  

**Evening:**  
- **Dinner:** Try kaiseki (multi-course meal) 

## 3. Best Practices & Considerations

1. **Reasoning Handling**: Use regex to separate <think> content from final answers
2. **Safety**: Built-in content filtering - handle HttpResponseError for violations
3. **Performance**:
   - Max tokens: 4096
   - Rate limit: 200K tokens/minute
4. **Cost**: Pay-as-you-go with serverless deployment
5. **Streaming**: Implement response streaming for long completions

```python
# Streaming example
response = client.complete(..., stream=True)
for chunk in response:
    print(chunk.choices[0].delta.content or "", end="")
```

## 🎯 Key Takeaways
- Leverage 128K context for detailed analysis
- Extract reasoning steps for debugging/analysis
- Combine with Azure AI Content Safety for production
- Monitor token usage via response.usage

> Always validate model outputs for critical applications!